# **Model Evaluation**

## Installation of Required Libraries

In this step, I install the necessary libraries for the evaluation.  
I use:
- `evaluate` for BLEU and ROUGE metrics  
- `rouge_score` for ROUGE computation  
- `sacrebleu` for standardized BLEU scoring

In [ ]:
!pip install evaluate rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=874afda5f77e8b4a3a0c292dd79d0129e23ae496fd30cda102331ff22cc80bf8
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


## Importing Libraries

Here, I import all required Python libraries for data processing and evaluation:
- `pandas` for handling CSV files  
- `evaluate` for computing evaluation metrics  

In [ ]:
import pandas as pd
import evaluate

bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

def clean_df(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.replace(";", "", regex=False)
        .str.strip()
        .str.lower()
    )
    return df

def normalize_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.strip()
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Loading the Reference Dataset

I load the reference dataset, which contains the correct answers used for evaluation.  

I clean the column names and rename `correct_answer` to `reference`.  
Additionally, I normalize the text fields to ensure consistent matching between predictions and references.

In [ ]:
def load_solutions(path):
    df = pd.read_csv(path, sep=";", engine="python", dtype=str)

    # Spalten säubern
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
    )

    df = df.rename(columns={
        "correct_answer": "reference"
    })

    # Clean
    df["id"] = df["id"].astype(str).str.strip()
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["reference"] = df["reference"].astype(str).str.strip()

    # 🔥 HIER: prompt behalten!
    return df[["id", "prompt", "reference"]]

## Loading Model Outputs

I load the outputs of each model, which contain predictions in the format:
- `id`
- `answer`

I clean the data and rename `answer` to `prediction` for consistency.

Since some CSV files have formatting issues, I apply additional preprocessing to correctly parse the data.

In [ ]:
def load_model(path):
    # Datei einlesen (alles in eine Spalte)
    df = pd.read_csv(
        path,
        sep=";",              # erstmal egal
        engine="python",
        dtype=str,
        on_bad_lines="skip"
    )

    # Spalten säubern
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
    )


    # FALL: alles in "id,answer"
    if "id,answer" in df.columns:
        split = df["id,answer"].str.split(",", n=1, expand=True)
        split.columns = ["id", "prediction"]
        df = split

    else:
        # normaler Fall
        df = df.rename(columns={"answer": "prediction"})

    # Clean
    df["id"] = df["id"].astype(str).str.strip()
    df["prediction"] = df["prediction"].astype(str).str.strip()


    return df[["id", "prediction"]]

## Model Evaluation

I evaluate each model by comparing its predictions with the reference answers.

For Model 1:
- I perform matching using the `prompt` field, as no reliable IDs are available.

For Model 2 and Model 3:
- I perform matching using the `id` field.

I compute the following evaluation metrics:
- BLEU (measures lexical similarity)  
- ROUGE-1 (measures word overlap)  
- ROUGE-L (measures sequence similarity)

In [ ]:
def evaluate_model(model_path, solutions_df):

    # Model 1 → prompt matching
    if "model1" in model_path:
        pred_df = load_model(model_path)

        # id ist eigentlich prompt
        pred_df = pred_df.rename(columns={"id": "prompt"})
        pred_df["prompt"] = pred_df["prompt"].astype(str).str.strip()

        df = pred_df.merge(
            solutions_df[["prompt", "reference"]],
            on="prompt",
            how="inner"
        )

    # Model 2 & 3 → id matching
    else:
        pred_df = load_model(model_path)

        pred_df["id"] = pred_df["id"].astype(str).str.strip()
        solutions_df["id"] = solutions_df["id"].astype(str).str.strip()

        df = pred_df.merge(
            solutions_df[["id", "reference"]],
            on="id",
            how="inner"
        )


    if df.empty:
        raise ValueError(f"Kein Match für {model_path}")

    predictions = df["prediction"].tolist()
    references = df["reference"].tolist()

    # BLEU
    bleu_score = bleu.compute(
        predictions=predictions,
        references=[[r] for r in references]
    )["score"]

    # ROUGE
    rouge_score = rouge.compute(
        predictions=predictions,
        references=references
    )


    return {
        "N": len(df),
        "BLEU": bleu_score,
        "ROUGE-1": rouge_score["rouge1"],
        "ROUGE-L": rouge_score["rougeL"]
    }

## Running the Evaluation

In this step, I evaluate all three models and collect the results in a table.

Each row represents a model, and each column represents an evaluation metric.

In [ ]:
results = {}

results["Model 1"] = evaluate_model("model1_outputs.csv", solutions_df)
results["Model 2"] = evaluate_model("model2_outputs.csv", solutions_df)
results["Model 3"] = evaluate_model("model3_outputs.csv", solutions_df)

results_df = pd.DataFrame(results).T
results_df

Raw columns: ['id,answer', 'unnamed: 1']
Fixed columns: ['id', 'prediction']
ID sample: ['Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?', 'Welche steuerlichen Konsequenzen hat es', 'Welche Körperschaften sind verpflichtet', 'Wie wird eine Dividende einer österreichischen Tochtergesellschaft (a) bei der Tochter und (b) bei der österreichischen Muttergesellschaft steuerlich behandelt?', 'nan']
model1_outputs.csv → Matches: 335
Raw columns: ['id,answer']
Fixed columns: ['id', 'prediction']
ID sample: ['CORP-TAX-001', 'CORP-TAX-002', 'CORP-TAX-003', 'CORP-TAX-004', 'CORP-TAX-005']
model2_outputs.csv → Matches: 643
Raw columns: ['id,answer']
Fixed columns: ['id', 'prediction']
ID sample: ['CORP-TAX-001', 'CORP-TAX-002', 'Firstly', 'Secondly', 'CORP-TAX-003']
model3_outputs.csv → Matches: 643


,N,BLEU,ROUGE-1,ROUGE-L
Model 1,335.0,4.657051,0.228270,0.170585
Model 2,643.0,1.115224,0.107845,0.079031
Model 3,643.0,0.494379,0.025762,0.021951
